# Two periodograms of one light curve

A reviewed notebook for the Gravitas compute page. It reads one `gravitas.observation/1` time series, handed to it as the text `gravitas_input`, and its two settings as the text `gravitas_params`. The example is the TESS light curve of HD 209458 that the Observatory's box search reads. It computes the two periodograms astronomers reach for first, with Astropy, and puts them side by side:

- **Lomb-Scargle** (`astropy.timeseries.LombScargle`): the power of the best sinusoid at each trial frequency, weighted by the error bars. It was made for unevenly spaced data and for signals that are sinusoids: a spotted star's rotation, a pulsation, a binary's ellipsoidal light.
- **Box least squares** (`astropy.timeseries.BoxLeastSquares`): the depth, at each trial period, of the best box, which is what a transit looks like. The Observatory's box search is the same idea (Kovacs, Zucker and Mazeh 2002).

What it does, in order:

1. **Read.** The time column is converted to a barycentric Julian date on the TDB scale with `astropy.time.Time`, so the epoch can be stated in the unit the literature uses.
2. **Lomb-Scargle** on a grid uniform in frequency, ten samples per peak, from the longest period to 0.5 day. The best period, its power and its false-alarm probability (the naive formula, `1 - (1 - p)^N` for `N` independent frequencies, because Baluev's needs SciPy, which this notebook does not download).
3. **Box least squares** on a grid uniform in frequency whose step is set by the shortest duration, with durations from one hour to the setting, then a fine grid around the best peak scored by the likelihood. Its period, epoch, duration and depth with a signal-to-noise ratio. The depth's error bar, and the signal-to-noise ratio made from it, are rescaled by the box model's reduced chi-square, because the data's scatter is larger than its error bars.
4. **Compare.** The ratio of the two best periods, and a table of both periodograms on one log-period axis (the larger of the values in each bin).

What the comparison shows: a sinusoid is a poor description of a transit, so Lomb-Scargle's tallest peak is not at the planet's period. And a box search has no formal uncertainty: its period is the best point of a grid, and how sharp its peak is depends on the grid, the durations and the box's shape. The notebook says so and does not invent an error bar. A transit is a few dips, and no periodogram can tell the planet's period from its half or its double without looking at the dips.

In [ ]:
# Read the input and the two settings.
import json, hashlib
import numpy as np
from astropy.time import Time
from astropy.timeseries import BoxLeastSquares, LombScargle
from astropy.utils import iers

# Nothing here needs a leap-second or Earth-orientation update from the network.
iers.conf.auto_download = False
iers.conf.auto_max_age = None

obs = json.loads(gravitas_input)
params = json.loads(gravitas_params)
assert obs["format"] == "gravitas.observation" and obs["formatVersion"] == 1, "this notebook reads gravitas.observation/1"
assert obs["kind"] == "time-series", "this notebook reads a time series"
MAXP = float(params.get("max_period", 15))
MAXD = float(params.get("max_duration", 6))
assert 5 <= MAXP <= 20, "max_period is between 5 and 20 days"
assert 2 <= MAXD <= 8, "max_duration is between 2 and 8 hours"
input_digest = hashlib.sha256(gravitas_input.encode("utf-8")).hexdigest()

def column(role, of=None):
    for c in obs["columns"]:
        if c["role"] == role and (of is None or c.get("of") == of):
            return c
    return None

xc = next(c for c in obs["columns"] if c["id"] == obs["time"]["column"])
assert xc["unit"] == "d", "the time column must be in days"
yc = column("value")
ec = column("uncertainty", of=yc["id"])
assert ec is not None, "the light curve needs an uncertainty column"
num = lambda c: np.array([np.nan if v is None else v for v in c["values"]], float)
t, f, e = num(xc), num(yc), num(ec)
masked = np.zeros(len(t), bool)
for m in obs["masks"]:
    masked[m["rows"]] = True
keep = np.isfinite(t) & np.isfinite(f) & np.isfinite(e) & (e > 0) & ~masked
t, f, e = t[keep], f[keep], e[keep]
level = np.median(f)
y, dy = f / level, e / level
span = float(t.max() - t.min())
assert span > 2 * 0.5, "the light curve is shorter than the shortest period searched"
# BTJD is BJD - 2457000 on the TDB scale; Time says what that is as a date.
offset = 2457000.0 if obs["time"]["format"] == "BTJD" else 0.0
first = Time(t.min() + offset, format="jd", scale="tdb")
print(f"{len(t)} points over {span:.2f} d, from {first.iso[:10]} (barycentric Julian date {first.jd:.3f}, {obs['time']['scale']})")

In [ ]:
# Lomb-Scargle: the best sinusoid at each frequency, weighted by the error bars.
ls = LombScargle(t, y, dy)
freq, power = ls.autopower(minimum_frequency=1.0 / MAXP, maximum_frequency=1.0 / 0.5, samples_per_peak=10)
top = int(np.argmax(power))
ls_period, ls_power = float(1.0 / freq[top]), float(power[top])
ls_fap = float(ls.false_alarm_probability(ls_power, method="naive", minimum_frequency=1.0 / MAXP, maximum_frequency=1.0 / 0.5))
print(f"Lomb-Scargle: best period {ls_period:.4f} d, power {ls_power:.3f}, false-alarm probability {ls_fap:.2g}; {len(freq)} frequencies")

In [ ]:
# Box least squares: the best box at each trial period.
durations = np.arange(1.0, MAXD + 0.5, 1.0) / 24.0
bls = BoxLeastSquares(t, y, dy)
grid = np.asarray(bls.autoperiod(durations, minimum_period=0.5, maximum_period=MAXP, frequency_factor=1.0), float)
coarse = bls.power(grid, durations, objective="likelihood")
top = int(np.argmax(coarse.power))
step = float(np.median(np.abs(np.diff(grid))))
# A fine grid across the best peak, with finer durations.
fine_periods = np.linspace(coarse.period[top] - 5 * step, coarse.period[top] + 5 * step, 2001)
fine_durations = np.linspace(durations[0], durations[-1], 21)
fine = bls.power(fine_periods, fine_durations, objective="likelihood")
j = int(np.argmax(fine.power))
bls_period, bls_duration = float(fine.period[j]), float(fine.duration[j])
bls_epoch, bls_depth = float(fine.transit_time[j]), float(fine.depth[j])
resolution = float(fine_periods[1] - fine_periods[0])
model = bls.model(t, bls_period, bls_duration, bls_epoch)
reduced = float((((y - model) / dy) ** 2).sum() / (len(t) - 4))
depth_sigma = float(fine.depth_err[j]) * np.sqrt(max(1.0, reduced))
bls_snr = bls_depth / depth_sigma
print(f"Box least squares: period {bls_period:.5f} d (grid step {step:.2g}, fine step {resolution:.1g}), duration {24 * bls_duration:.2f} h, depth {1e6 * bls_depth:.0f} ppm, S/N {bls_snr:.0f}; {len(grid)} periods")

In [ ]:
# Put the two on one axis: the larger value of each periodogram in each of 400 bins of log period.
edges = np.exp(np.linspace(np.log(0.5), np.log(MAXP), 401))
centers = np.sqrt(edges[:-1] * edges[1:])
def binned(periods, values):
    which = np.clip(np.digitize(periods, edges) - 1, 0, 399)
    out = np.full(400, np.nan)
    for i, v in zip(which, values):
        if not v <= out[i]:
            out[i] = v
    return out
ls_binned = binned(1.0 / freq, power)
bls_binned = binned(coarse.period, coarse.depth_snr)
have = np.isfinite(ls_binned) & np.isfinite(bls_binned)
ratio = ls_period / bls_period
print(f"the two best periods differ by a factor {ratio:.4f}; {int(have.sum())} bins hold both")

In [ ]:
# What the notebook returns: an envelope and a table.
LICENSE = obs["license"]
epoch_bjd = Time(bls_epoch + offset, format="jd", scale="tdb")
envelope = {
    "format": "gravitas.artifact", "formatVersion": 1,
    "id": f"compute:periodograms:{input_digest[:16]}",
    "made": {"app": "gravitas"},
    "source": {"kind": "analysis", "id": "compute/periodograms", "version": "1", "digest": input_digest},
    "provenance": {
        "credit": obs["credit"], "license": LICENSE, "retrieved": obs["retrieved"],
        "citations": [c["text"] for c in obs["citations"]],
        "reductions": obs["reductions"] + [
            "Normalized by the median. Lomb-Scargle with the error bars as weights (astropy.timeseries.LombScargle), ten samples per peak between 0.5 day and the longest period; false-alarm probability by the naive formula for the number of independent frequencies.",
            f"Box least squares (astropy.timeseries.BoxLeastSquares), durations 1 to {MAXD:g} hours, a coarse grid then a fine grid across the best peak, scored by the likelihood; the epoch of the best box is barycentric Julian date {epoch_bjd.jd:.5f} (TDB).",
            "The depth's sigma is the box search's, rescaled by the box model's reduced chi-square when it exceeds 1; the signal-to-noise ratio is the depth over that sigma.",
        ],
    },
    "quantities": [
        {"id": "ls.period", "value": ls_period, "unit": "d", "origin": "measured", "uncertainty": {"kind": "none"}},
        {"id": "ls.power", "value": ls_power, "unit": "", "origin": "measured", "uncertainty": {"kind": "none"}},
        {"id": "ls.fap", "value": ls_fap, "unit": "", "origin": "derived", "uncertainty": {"kind": "none"}},
        {"id": "bls.period", "value": bls_period, "unit": "d", "origin": "measured", "uncertainty": {"kind": "none"}},
        {"id": "bls.epoch", "value": bls_epoch, "unit": "d", "origin": "measured", "uncertainty": {"kind": "none"}},
        {"id": "bls.duration", "value": 24 * bls_duration, "unit": "h", "origin": "measured", "uncertainty": {"kind": "none"}},
        {"id": "bls.depth", "value": 1e6 * bls_depth, "unit": "ppm", "origin": "measured", "uncertainty": {"kind": "sigma", "sigma": 1e6 * depth_sigma, "basis": "scaled"}},
        {"id": "bls.snr", "value": bls_snr, "unit": "", "origin": "derived", "uncertainty": {"kind": "none"}},
        {"id": "period.ratio", "value": float(ratio), "unit": "", "origin": "derived", "uncertainty": {"kind": "none"}},
    ],
    "warnings": [
        "a periodogram's best period has no error bar here: it is the best point of a grid, and the peak's sharpness depends on the grid, the durations and the box's shape. The BLS period is good to about the fine grid's step (" + f"{resolution:.1g} d" + ") and no better",
        "Lomb-Scargle fits sinusoids: a transit is not one, so its tallest peak is not the planet's period, and the false-alarm probability says there is a periodic signal and not that the period is right",
        "a periodogram cannot tell the planet's period from half or twice it: the dips have to be counted",
        "the epoch is in the light curve's own time (" + (obs["time"].get("reference") or obs["time"]["format"]) + ")",
    ],
}
table = {
    "format": "gravitas.observation", "formatVersion": 1, "kind": "table",
    "id": f"{obs['id']}:periodograms", "title": f"{obs['title']}: Lomb-Scargle and box least squares on one period axis",
    "object": obs["object"], "facility": obs["facility"], "origin": "model",
    "source": {"kind": "import", "id": obs["id"], "version": obs["source"].get("version")},
    "credit": obs["credit"], "license": LICENSE, "retrieved": obs["retrieved"],
    "citations": obs["citations"],
    "reductions": envelope["provenance"]["reductions"],
    "axes": {"x": "period", "y": "bls"},
    "columns": [
        {"id": "period", "name": "period (log bins, centers)", "unit": "d", "role": "x", "values": [float(v) for v in centers[have]]},
        {"id": "ls", "name": "Lomb-Scargle power", "unit": "", "role": "value", "values": [float(v) for v in ls_binned[have]]},
        {"id": "bls", "name": "box least squares depth S/N", "unit": "", "role": "value", "values": [float(v) for v in bls_binned[have]]},
    ],
    "masks": [], "annotations": [],
}
gravitas_output = {"envelope": envelope, "table": table}
print(len(table["columns"][0]["values"]), "rows returned")